# 23.07 - Inference speed

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Inference timing note.

**Priority:** P0 — runtime matters.

Build a small, repeatable image-inference pipeline and measure it before deciding which optimization helps.

## Core Ideas

- Call `model.eval()` so dropout and batch normalization use inference behavior.
- Use `torch.inference_mode()` to avoid autograd bookkeeping during prediction.
- Process examples in batches: tiny batches waste compute, while oversized batches can exhaust memory.
- Cache deterministic preprocessing such as resizing. For video, the same principle means avoiding repeated decoding of unchanged frames.
- Warm up the pipeline, synchronize accelerator work when needed, and compare identical inputs.
- Record both latency and throughput. The fastest setting must still produce the correct output shape and values.

## Setup and Prepared Image Data

The toy model and images keep this notebook offline and quick to run.

**Return structure — `TinyVisionModel(...)` and `TinyVisionModel.forward(images)`:**

- Constructing `TinyVisionModel(num_classes)` returns an `nn.Module` instance.
- Calling the instance returns a `torch.Tensor` of shape `[N, K]`, dtype `torch.float32`, on the input/model device.
- `N` is the batch size and `K` is `num_classes`; each row contains class logits.

In [1]:
import time
import torch
from torch import nn

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TinyVisionModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(8, num_classes)

    def forward(self, images):
        features = self.features(images).flatten(1)
        return self.classifier(features)


toy_images = torch.rand(24, 3, 32, 32, dtype=torch.float32, device=DEVICE)
model = TinyVisionModel(num_classes=4).to(DEVICE)
model.train()

print("device:", DEVICE)
print("prepared images:", toy_images.shape, toy_images.dtype)

device: cpu
prepared images: torch.Size([24, 3, 32, 32]) torch.float32


## Exercise 23-A: Split inputs into batches

Implement deterministic, order-preserving batching without copying examples to another device.

**Return structure — `make_batches(images, batch_size)`:**

- Returns a `list[torch.Tensor]` of length `ceil(N / batch_size)`.
- Each item has shape `[B_i, C, H, W]`, the same dtype and device as `images`.
- All full items have `B_i == batch_size`; the final item has `1 <= B_i <= batch_size`.
- Concatenating the list on dimension 0 reconstructs the input order.

In [2]:
# TODO 23-A
import math
def make_batches(images, batch_size):
    num_batch = math.ceil(len(images) / batch_size)
    batches = []
    k = 0
    for i in range(num_batch) : 
        batches.append(images[k:k+batch_size,:,:,:])
        k += batch_size
    return batches

# Smoke check: run this after implementing the function above.
smoke_batches = make_batches(toy_images[:10], batch_size=4)
print("smoke batch sizes:", [batch.shape[0] for batch in smoke_batches])

smoke batch sizes: [4, 4, 2]


## Exercise 23-B: Predict safely in batches

Switch to evaluation behavior, disable gradient tracking, concatenate predictions, and restore the model's original training state.

**Return structure — `predict_batched(model, images, batch_size)`:**

- Returns one `torch.Tensor` of shape `[N, K]` on the model/input device.
- Its dtype is the model output dtype (`torch.float32` here), and `requires_grad` is `False`.
- Row order matches `images`.
- The function restores `model.training` to its value from before the call.

In [6]:
# TODO 23-B
def predict_batched(model, images, batch_size):
    # Use make_batches(), model.eval(), and torch.inference_mode().
    batches = make_batches(images, batch_size)
    training = model.training
    model.eval()
    logits = []
    with torch.inference_mode() :
        for batch in batches : 
            logit = model(batch)
            logits.append(logit)
        logits = torch.cat(logits,dim = 0)
    model.train(training)
    return logits
        


# Smoke check: run this after implementing the function above.
model.train()
smoke_logits = predict_batched(model, toy_images[:7], batch_size=3)
print("smoke logits:", smoke_logits.shape, "requires_grad=", smoke_logits.requires_grad)
print("training state restored:", model.training)

smoke logits: torch.Size([7, 4]) requires_grad= False
training state restored: True


## Exercise 23-C: Cache deterministic resizing

Resize once per requested size and reuse the resulting tensors. In a real video pipeline, cache sampled or decoded frames only when memory permits.

**Return structure — `build_resize_cache(images, sizes)`:**

- Returns a `dict[str, torch.Tensor]` with one entry per requested `(height, width)` tuple.
- Each key is formatted as `"HxW"`, for example `"16x20"`.
- Each value has shape `[N, C, H, W]`, dtype `torch.float32`, and the same device as `images`.
- Values are bilinearly resized tensors; input order is unchanged.

In [7]:
# TODO 23-C
def build_resize_cache(images, sizes):
    cache = {}
    for h,w in sizes : 
        key = str(h) + "x" + str(w)
        resized = torch.nn.functional.interpolate(images, (h,w), mode = "bilinear")
        cache[key] = resized
    return cache


# Smoke check: run this after implementing the function above.
smoke_cache = build_resize_cache(toy_images[:5], [(16, 16), (24, 20)])
print("smoke cache:", {key: tuple(value.shape) for key, value in smoke_cache.items()})

smoke cache: {'16x16': (5, 3, 16, 16), '24x20': (5, 3, 24, 20)}


## Exercise 23-D: Benchmark a callable

Warm-up runs are excluded from the measurement. Synchronization is needed because accelerator kernels can finish after Python continues.

**Return structure — `benchmark_callable(callable_fn, warmup, repeats)`:**

- Returns a dictionary with exactly three keys.
- `"runs_seconds"`: `list[float]` of length `repeats`, one non-negative elapsed time per run.
- `"mean_seconds"`: `float`, the arithmetic mean of the recorded runs.
- `"min_seconds"`: `float`, the smallest recorded run.
- The callable's own return value is intentionally discarded.

In [8]:
# TODO 23-D
def benchmark_callable(callable_fn, warmup=1, repeats=3):
    if warmup < 0 or repeats <= 0:
        raise ValueError("warmup must be non-negative and repeats must be positive")

    for _ in range(warmup):
        callable_fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    runs_seconds = []
    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        start = time.perf_counter()
        callable_fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        runs_seconds.append(time.perf_counter() - start)

    return {
        "runs_seconds": runs_seconds,
        "mean_seconds": sum(runs_seconds) / len(runs_seconds),
        "min_seconds": min(runs_seconds),
    }


# Smoke check: run this after implementing the function above.
smoke_timing = benchmark_callable(
    lambda: predict_batched(model, toy_images[:8], batch_size=4),
    warmup=1,
    repeats=2,
)
print("smoke timing:", smoke_timing)

smoke timing: {'runs_seconds': [0.0005849000299349427, 0.0004984999541193247], 'mean_seconds': 0.0005416999920271337, 'min_seconds': 0.0004984999541193247}


## Exercise 23-E: Profile candidate batch sizes

Compare several batch sizes on the exact same images and model. Keep correctness evidence beside timing evidence.

**Return structure — `profile_batch_sizes(model, images, batch_sizes, repeats)`:**

- Returns a `list[dict]` with one item per requested batch size, in input order.
- Every dictionary has exactly four keys:
  - `"batch_size"`: positive `int`.
  - `"mean_seconds"`: non-negative `float`.
  - `"images_per_second"`: positive `float`.
  - `"output_shape"`: two-item `tuple[int, int]` equal to `(N, K)`.
- The function returns timing records only and does not change the model's original training state.

In [9]:
# TODO 23-E
def profile_batch_sizes(model, images, batch_sizes, repeats=2):
    report = []
    for batch_size in batch_sizes:
        predictor = lambda selected_size=batch_size: predict_batched(model, images, selected_size)
        timing = benchmark_callable(predictor, warmup=1, repeats=repeats)
        logits = predictor()
        mean_seconds = timing["mean_seconds"]
        report.append({
            "batch_size": batch_size,
            "mean_seconds": mean_seconds,
            "images_per_second": images.shape[0] / max(mean_seconds, 1e-12),
            "output_shape": tuple(logits.shape),
        })
    return report


# Smoke check: run this after implementing the function above.
smoke_report = profile_batch_sizes(model, toy_images[:12], [1, 4, 12], repeats=2)
for row in smoke_report:
    print(row)

{'batch_size': 1, 'mean_seconds': 0.001510749978478998, 'images_per_second': 7943.074744956431, 'output_shape': (12, 4)}
{'batch_size': 4, 'mean_seconds': 0.0008862499962560833, 'images_per_second': 13540.197518412831, 'output_shape': (12, 4)}
{'batch_size': 12, 'mean_seconds': 0.0004146499850321561, 'images_per_second': 28940.070983167647, 'output_shape': (12, 4)}


## Test Cases

Run this cell after completing all TODO cells.

**Return structure — `run_day23_tests()`:**

- Returns `None`.
- Success is communicated by completing all assertions and printing exactly `Day 23 tests passed`.

In [10]:
def run_day23_tests():
    assert "make_batches" in globals(), "Missing function: make_batches"
    assert "predict_batched" in globals(), "Missing function: predict_batched"
    assert "build_resize_cache" in globals(), "Missing function: build_resize_cache"
    assert "benchmark_callable" in globals(), "Missing function: benchmark_callable"
    assert "profile_batch_sizes" in globals(), "Missing function: profile_batch_sizes"

    sample = toy_images[:13]
    batches = make_batches(sample, batch_size=5)
    assert [batch.shape[0] for batch in batches] == [5, 5, 3]
    assert torch.equal(torch.cat(batches, dim=0), sample)
    assert all(batch.dtype == sample.dtype and batch.device == sample.device for batch in batches)

    model.train()
    logits = predict_batched(model, sample, batch_size=5)
    assert logits.shape == (13, 4)
    assert logits.dtype == torch.float32 and logits.device == sample.device
    assert not logits.requires_grad
    assert model.training, "predict_batched must restore the original training state"
    model.eval()
    with torch.inference_mode():
        direct_logits = model(sample)
    model.train()
    assert torch.allclose(logits, direct_logits, atol=1e-6)

    cache = build_resize_cache(sample, [(16, 16), (20, 24)])
    assert set(cache) == {"16x16", "20x24"}
    assert cache["16x16"].shape == (13, 3, 16, 16)
    assert cache["20x24"].shape == (13, 3, 20, 24)
    assert all(value.dtype == sample.dtype and value.device == sample.device for value in cache.values())

    timing = benchmark_callable(lambda: torch.ones(1, device=DEVICE) + 1, warmup=0, repeats=2)
    assert set(timing) == {"runs_seconds", "mean_seconds", "min_seconds"}
    assert len(timing["runs_seconds"]) == 2
    assert timing["mean_seconds"] >= 0 and timing["min_seconds"] >= 0

    model.train()
    report = profile_batch_sizes(model, sample[:4], [1, 4], repeats=1)
    assert len(report) == 2
    assert [row["batch_size"] for row in report] == [1, 4]
    assert all(set(row) == {"batch_size", "mean_seconds", "images_per_second", "output_shape"} for row in report)
    assert all(row["output_shape"] == (4, 4) for row in report)
    assert all(row["images_per_second"] > 0 for row in report)
    assert model.training, "profiling must restore the original training state"

    print("Day 23 tests passed")


run_day23_tests()

Day 23 tests passed


## Day 23 Checklist

- [ ] I can explain `eval()` versus `inference_mode()`.
- [ ] I preserve input order when batching predictions.
- [ ] I restore the model's original train/eval state.
- [ ] I cache deterministic preprocessing instead of repeating it.
- [ ] I warm up and synchronize before trusting timings.
- [ ] I compare throughput without dropping correctness checks.
- [ ] I recorded the best batch size and one remaining bottleneck in my inference timing note.